# KG2RAG-lite

This notebook demonstrates a hybrid retrieval system that combines dense retrieval and knowledge graph traversal for entity-centric question answering on cultural heritage data.

Pipeline:

Query \
↓ \
Dense Retrieval \
↓ \
Entity Extraction \
↓ \
Relation-aware KG Traversal \
↓ \
Hybrid Reranking \
↓ \
Answer Generation \
\
Code flow: \
\
1 Setup \
 \
2 Data preprocessing \
 \
3 Knowledge Graph construction \
 \
4 Chunk generation \
 \
5 Dense retrieval (Embeddings) \
 \
6 Entity extraction \
 \
7 KG traversal \
 \
8 Hybrid reranking \
 \
9 Structured answer 

## 1. Setup

In [1]:
import os
import sys
import random
from time import time

import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from argparse import Namespace

import re

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict, deque
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [2]:
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")

In [3]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

## 2. Data preprocessing

In [4]:
def clean_value(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    if x == "" or x.lower() == "nan" or x=='[]':
        return None
    return x

def split_multi_value(x):
    if x is None:
        return []
    parts = [p.strip() for p in re.split(r"[,\uFF0C]", x) if p.strip()]
    return parts

def normalize_author(name):

    if name is None:
        return None

    name = str(name).strip()

    # 괄호 + 한자 제거
    name = re.sub(r"\(.*?\)", "", name)

    # "전 " 제거
    name = re.sub(r"^전\s+", "", name)

    # 공백 정리
    name = re.sub(r"\s+", " ", name)

    return name.strip()

In [5]:
data = 'datasample.xlsx'
sheets = ['Sheet1']
df_list = [pd.read_excel(data, sheet) for sheet in sheets]
df = pd.concat(df_list, ignore_index=True)

In [6]:
USE_COLS = [
    "소장품번호", "국적", "시대", '유물',
    "유물분류 3", "유물분류 4",
    "문양", "재질 1", "재질 2",
    "작가명", "출생년도"
]

REL_MAP = {
    "유물": "has_name",
    "국적": "has_country",
    "시대": "has_era",
    "유물분류 3": "has_category3",
    "유물분류 4": "has_category4",
    "문양": "has_pattern",
    "재질 1": "has_material1",
    "재질 2": "has_material2",
    "작가명": "created_by",
    "출생년도": "creator_birth_year",
}

In [7]:
df = pd.concat(df_list, ignore_index=True)
data = df[USE_COLS].copy()

for col in USE_COLS:
    data[col] = data[col].apply(clean_value)

data['작가명'] = data['작가명'].apply(normalize_author)

## 3. Konwledge Graph Construction

In [8]:
# triple 생성

triples = []

for _, row in data.iterrows():
    head = row["소장품번호"]
    if head is None:
        continue

    for col, rel in REL_MAP.items():
        val = row[col]
        if val is None:
            continue

        if col == "문양":
            for tail in split_multi_value(val):
                triples.append({
                    "head": head,
                    "relation": rel,
                    "tail": tail
                })
        else:
            triples.append({
                "head": head,
                "relation": rel,
                "tail": val
            })

triples_df = pd.DataFrame(triples)

In [9]:
# entity 생성

entity_set = set()

for col in [
    "소장품번호",
    "유물",
    "국적",
    "시대",
    "유물분류 3",
    "유물분류 4",
    "재질 1",
    "재질 2",
    "작가명",
    "출생년도"
]:
    vals = data[col].dropna().astype(str).str.strip().tolist()
    entity_set.update([v for v in vals if v])

for val in data["문양"].dropna().tolist():
    entity_set.update(split_multi_value(val))

entity_list = sorted(entity_set, key=lambda x: len(str(x)), reverse=True)


## 4. Make Chunks

In [10]:
def make_chunk(row):

    sents = []

    item_id = row["소장품번호"]
    name = row["유물"]
    country = row["국적"]
    era = row["시대"]
    c3 = row["유물분류 3"]
    c4 = row["유물분류 4"]
    pattern = row["문양"]
    m1 = row["재질 1"]
    m2 = row["재질 2"]
    author = row["작가명"]
    birth = row["출생년도"]

    # 첫 문장 (유물명 포함)
    sent1_parts = []

    if name:
        sent1_parts.append(f"{name}")

    if item_id:
        sent1_parts.append(f"(소장품번호 {item_id})")

    if country:
        sent1_parts.append(f"{country}의")

    if era:
        sent1_parts.append(f"{era} 시대")

    sent1_parts.append("유물이다.")

    sents.append(" ".join(sent1_parts))

    # 분류
    categories = [x for x in [c3, c4] if x]
    if categories:
        sents.append(f"유물분류는 {', '.join(categories)}이다.")

    # 문양
    if pattern:
        sents.append(f"문양은 {pattern}이다.")

    # 재질
    materials = [x for x in [m1, m2] if x]
    if materials:
        sents.append(f"재질은 {', '.join(materials)}이다.")

    # 작가 / 출생년도
    if author and birth:
        sents.append(f"작가는 {author}이며 출생년도는 {birth}년이다.")
    elif author:
        sents.append(f"작가는 {author}이다.")
    elif birth:
        sents.append(f"출생년도는 {birth}년이다.")

    return " ".join(sents)

In [11]:
# chunk 생성

chunks = []

for _, row in data.iterrows():

    item_id = row["소장품번호"]

    if item_id is None:
        continue

    text = make_chunk(row)

    chunks.append({
        "chunk_id": f"chunk_{item_id}",
        "item_id": item_id,
        "text": text
    })

chunks_df = pd.DataFrame(chunks)

## 5. Dense Retrieval (embeddings)

In [12]:
def dense_retrieve(query, top_k=5):
    q_emb = embed_model.encode([query], convert_to_numpy=True)
    sims = cosine_similarity(q_emb, chunk_embeddings)[0]

    top_indices = np.argsort(sims)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            "chunk_id": chunk_ids[idx],
            "item_id": chunk_item_ids[idx],
            "text": chunk_texts[idx],
            "dense_score": float(sims[idx])
        })
    return results, sims

In [13]:
embed_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# chunk embedding

chunk_texts = chunks_df["text"].tolist()
chunk_ids = chunks_df["chunk_id"].tolist()
chunk_item_ids = chunks_df["item_id"].tolist()

chunk_embeddings = embed_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

# chunk_id -> index
chunk_id_to_idx = {cid: i for i, cid in enumerate(chunk_ids)}
item_id_to_chunk_id = dict(zip(chunks_df["item_id"], chunks_df["chunk_id"]))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

## 6. Entities Extraction

In [14]:
def extract_entities(text, entity_list):
    text = str(text)
    found = []
    for ent in entity_list:
        if ent and ent in text:
            found.append(ent)
    # 중복 제거
    return list(dict.fromkeys(found))

## 7. Retrival Support Mapping

In [15]:
#KG adjacency

# item -> [(relation, tail)]
forward_adj = defaultdict(list)

# tail(entity) -> [(relation, head)]
reverse_adj = defaultdict(list)

for _, row in triples_df.iterrows():
    h = row["head"]
    r = row["relation"]
    t = row["tail"]

    forward_adj[h].append((r, t))
    reverse_adj[t].append((r, h))

In [16]:
# item별 연결 entity 집합
item_to_entities = defaultdict(set)

for _, row in triples_df.iterrows():
    item_to_entities[row["head"]].add(row["tail"])

item_rel_values = defaultdict(lambda: defaultdict(list))

for _, row in triples_df.iterrows():
    h = row["head"]
    r = row["relation"]
    t = row["tail"]

    if t not in item_rel_values[h][r]:
        item_rel_values[h][r].append(t)

# chunk_id -> entity들
chunk_to_entities = {}
for _, row in chunks_df.iterrows():
    cid = row["chunk_id"]
    item_id = row["item_id"]
    ents = set(item_to_entities.get(item_id, set()))
    ents.add(item_id)
    chunk_to_entities[cid] = ents

## 8. KG Traversal

## 9. Support Hybrid rerank

In [17]:
def traverse_from_seed_entities(seed_entities, max_hop=2, rel_weights=None):
    """
    relation-aware traversal
    seed_entities로부터 관련 item 후보를 찾고,
    relation intent에 따라 edge 가중치를 반영한다.
    """
    if rel_weights is None:
        rel_weights = REL_BASE_WEIGHT

    visited_entities = set(seed_entities)
    visited_items = set()

    queue = deque()
    for ent in seed_entities:
        queue.append((ent, 0))

    candidate_items = defaultdict(float)

    while queue:
        current_node, hop = queue.popleft()

        if hop >= max_hop:
            continue

        # reverse: entity -> item
        for rel, head_item in reverse_adj.get(current_node, []):
            visited_items.add(head_item)

            rel_w = rel_weights.get(rel, 1.0)
            hop_score = 1.0 / (hop + 1)

            # relation-aware graph score
            score = rel_w * hop_score
            candidate_items[head_item] += score

            # item에서 다시 forward로 확장
            for next_rel, next_entity in forward_adj.get(head_item, []):
                next_rel_w = rel_weights.get(next_rel, 1.0)

                if next_entity not in visited_entities:
                    visited_entities.add(next_entity)

                    queue.append((next_entity, hop + 1))

    return candidate_items

In [18]:
def normalize_text(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

In [19]:
REL_KEYWORDS = {
    "created_by": ["작가", "그린", "그렸다", "그린 작품", "저자", "화가"],
    "has_pattern": ["문양", "무늬", "패턴"],
    "has_material1": ["재질", "소재", "재료"],
    "has_material2": ["재질", "소재", "재료"],
    "has_era": ["시대", "시기", "연대"],
    "has_category3": ["분류", "유형", "종류", "회화", "서화", "일반회화"],
    "has_category4": ["분류", "유형", "종류", "회화", "서화", "일반회화"],
    #"has_name": ["작품", "유물", "이름", "명칭"],
}

In [20]:
def extract_relation_intents(query, rel_keywords=REL_KEYWORDS):
    """
    query 안에서 relation 관련 키워드를 찾아
    relation별 score를 반환
    """
    query = normalize_text(query)

    rel_scores = defaultdict(float)

    for rel, keywords in rel_keywords.items():
        for kw in keywords:
            if kw in query:
                # 단순 포함이면 +1
                rel_scores[rel] += 1.0

                # 조금 더 긴 키워드는 가중치 약간 추가
                if len(kw) >= 3:
                    rel_scores[rel] += 0.2

    return dict(rel_scores)

In [21]:
def get_main_relation_intents(query, rel_keywords=REL_KEYWORDS, top_k=3):
    rel_scores = extract_relation_intents(query, rel_keywords)
    if not rel_scores:
        return []

    sorted_rels = sorted(rel_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_rels[:top_k]

In [22]:
def build_relation_weights(query, query_entities,
                           rel_keywords=REL_KEYWORDS,
                           base_weight=None):

    if base_weight is None:
        base_weight = REL_BASE_WEIGHT.copy()
    else:
        base_weight = base_weight.copy()

    # 1️⃣ keyword 기반 relation intent
    rel_scores = extract_relation_intents(query, rel_keywords)

    for rel, score in rel_scores.items():
        if rel in base_weight:
            base_weight[rel] += 0.5 * score

    # 2️⃣ entity type 기반 relation 강화
    for ent in query_entities:
        if ent in author_entities:
            base_weight["created_by"] += 1.0

    return base_weight, rel_scores

In [23]:
def get_relation_bonus(item_id, rel_scores, item_to_relations):
    """
    query에서 감지된 relation intent와
    해당 item이 가진 relation이 얼마나 맞는지 보너스 계산
    """
    if not rel_scores:
        return 0.0

    item_rels = item_to_relations.get(item_id, set())
    bonus = 0.0

    for rel, score in rel_scores.items():
        if rel in item_rels:
            bonus += 0.1 * score

    return bonus

In [24]:
REL_BASE_WEIGHT = {
    #"has_name": 1.0,
    "has_country": 1.0,
    "has_era": 1.0,
    "has_category3": 1.0,
    "has_category4": 1.0,
    "has_pattern": 1.0,
    "has_material1": 1.0,
    "has_material2": 1.0,
    "created_by": 1.0,
    "creator_birth_year": 1.0,
}

author_entities = set(
    data["작가명"].dropna().astype(str).str.strip().tolist()
)

In [25]:
item_to_relations = defaultdict(set)

for _, row in triples_df.iterrows():
    item_to_relations[row["head"]].add(row["relation"])

## 10. Hybrid Retrieval

In [26]:
def hybrid_retrieve(query, seed_top_k=5, final_top_k=8, max_hop=2,
                    alpha=0.65, beta=0.30, gamma=0.05):
    """
    alpha: dense score weight
    beta : graph score weight
    gamma: relation bonus weight
    """

    # 0) relation intent extraction
    query_entities = extract_entities(query, entity_list)

    rel_weights, rel_scores = build_relation_weights(
        query,
        query_entities
    )

    # 1) dense retrieval
    seed_results, sims = dense_retrieve(query, top_k=seed_top_k)

    # 2) seed chunk들에서 entity 추출
    query_entities = extract_entities(query, entity_list)

    seed_entities = set(query_entities)
    seed_chunk_ids = set()

    # for res in seed_results:
    #     cid = res["chunk_id"]
    #     seed_chunk_ids.add(cid)

    #     # chunk text에서 entity 추출
    #     chunk_ents = extract_entities(res["text"], entity_list)
    #     seed_entities.update(chunk_ents)

    #     # 미리 연결된 entity도 추가
    #     seed_entities.update(chunk_to_entities.get(cid, set()))

    # 3) relation-aware KG traversal
    candidate_items = traverse_from_seed_entities(
        seed_entities,
        max_hop=max_hop,
        rel_weights=rel_weights
    )

    # 4) candidate item -> chunk 매핑
    candidate_chunk_ids = set(seed_chunk_ids)

    for item_id in candidate_items.keys():
        if item_id in item_id_to_chunk_id:
            candidate_chunk_ids.add(item_id_to_chunk_id[item_id])

    # 5) hybrid rerank
    results = []
    max_graph_score = max(candidate_items.values()) if len(candidate_items) > 0 else 1.0

    for cid in candidate_chunk_ids:
        idx = chunk_id_to_idx[cid]
        row = chunks_df.loc[chunks_df["chunk_id"] == cid].iloc[0]

        item_id = row["item_id"]
        text = row["text"]

        dense_score = float(sims[idx])

        raw_graph_score = candidate_items.get(item_id, 0.0)
        graph_score = raw_graph_score / max_graph_score if max_graph_score > 0 else 0.0

        relation_bonus = get_relation_bonus(item_id, rel_scores, item_to_relations)

        hybrid_score = (
            alpha * dense_score
            + beta * graph_score
            + gamma * relation_bonus
        )

        results.append({
            "chunk_id": cid,
            "item_id": item_id,
            "text": text,
            "dense_score": dense_score,
            "graph_score": graph_score,
            "relation_bonus": relation_bonus,
            "hybrid_score": hybrid_score
        })

    results = sorted(results, key=lambda x: x["hybrid_score"], reverse=True)[:final_top_k]

    return {
        "query": query,
        "query_entities": query_entities,
        "relation_intents": rel_scores,
        "relation_weights": rel_weights,
        "seed_entities": list(seed_entities),
        "seed_results": seed_results,
        "final_results": results
    }

## 11. Structure Answer Generation 

In [27]:
ANSWER_REL_PRIORITY = [
    "has_pattern",
    "has_material1",
    "has_material2",
    "has_era",
    "created_by",
    "creator_birth_year"
]

def choose_answer_relation(relation_intents):
    """
    query에서 추출된 relation intent 중
    실제 답변에 사용할 대표 relation을 고른다.
    """
    if not relation_intents:
        return None

    # relation_intents가 dict면 score 기준 정렬
    if isinstance(relation_intents, dict):
        sorted_rels = sorted(relation_intents.items(), key=lambda x: x[1], reverse=True)
        rels = [r for r, _ in sorted_rels]
    else:
        rels = list(relation_intents)

    # 우선순위 반영
    for rel in ANSWER_REL_PRIORITY:
        if rel in rels:
            return rel

    return rels[0] if rels else None

In [28]:
REL_TO_KOR = {
    "has_pattern": "문양",
    "has_material1": "재질",
    "has_material2": "재질",
    "has_era": "시대",
    "created_by": "작가",
    "creator_birth_year": "출생년도"
}

In [29]:
def build_structured_answer(query, retrieval_out, item_rel_values):
    """
    retrieval 결과에서 top item을 기준으로
    relation intent에 맞는 값을 꺼내서 답변 생성
    """

    final_results = retrieval_out.get("final_results", [])
    relation_intents = retrieval_out.get("relation_intents", {})
    query_entities = retrieval_out.get("query_entities", [])

    if not final_results:
        return {
            "answer": "관련된 결과를 찾지 못했습니다.",
            "top_item_id": None,
            "target_relation": None,
            "values": []
        }

    top_item = final_results[0]["item_id"]
    top_text = final_results[0]["text"]

    target_rel = choose_answer_relation(relation_intents)

    if target_rel is None:
        return {
            "answer": f"가장 관련 있는 결과는 다음과 같습니다: {top_text}",
            "top_item_id": top_item,
            "target_relation": None,
            "values": []
        }

    # 재질은 1,2를 같이 처리
    if target_rel in ["has_material1", "has_material2"]:
        values = []
        values.extend(item_rel_values[top_item].get("has_material1", []))
        values.extend(item_rel_values[top_item].get("has_material2", []))
        values = list(dict.fromkeys([v for v in values if v]))
        rel_name = "재질"
    else:
        values = item_rel_values[top_item].get(target_rel, [])
        rel_name = REL_TO_KOR.get(target_rel, target_rel)

    item_name_vals = item_rel_values[top_item].get("has_name", [])
    item_name = item_name_vals[0] if item_name_vals else top_item

    if not values:
        answer = f"{item_name}에 대해 요청한 {rel_name} 정보를 찾지 못했습니다."
    else:
        if rel_name == "문양":
            answer = f"{item_name}에 등장하는 문양은 {', '.join(values)}이다."
        elif rel_name == "재질":
            answer = f"{item_name}의 재질은 {', '.join(values)}이다."
        elif rel_name == "시대":
            answer = f"{item_name}의 시대는 {', '.join(values)}이다."
        elif rel_name == "작가":
            answer = f"{item_name}의 작가는 {', '.join(values)}이다."
        elif rel_name == "출생년도":
            answer = f"{item_name} 작가의 출생년도는 {', '.join(values)}년이다."
        else:
            answer = f"{item_name}의 {rel_name} 정보는 {', '.join(values)}이다."

    return {
        "answer": answer,
        "top_item_id": top_item,
        "target_relation": target_rel,
        "values": values
    }

## 12. Example

In [30]:
query = "이방운 작품에는 어떤 문양이 등장하는가?"
out = hybrid_retrieve(query, seed_top_k=5, final_top_k=5, max_hop=2)

print("Query entities:", out["query_entities"])
print("Relation intents:", out["relation_intents"])
print("Relation weights:", out["relation_weights"])


print("\n=== Seed Results ===")
for r in out["seed_results"]:
    print(f"[{r['chunk_id']}] dense={r['dense_score']:.4f}")
    print(r["text"])
    print()

print("\n=== Final Hybrid Results ===")
for r in out["final_results"]:
    print(
        f"[{r['chunk_id']}] hybrid={r['hybrid_score']:.4f} | "
        f"dense={r['dense_score']:.4f} | graph={r['graph_score']:.4f} | "
        f"rel_bonus={r['relation_bonus']:.4f}"
    )
    print(r["text"])
    print()

Query entities: ['이방운']
Relation intents: {'has_pattern': 1.0}
Relation weights: {'has_country': 1.0, 'has_era': 1.0, 'has_category3': 1.0, 'has_category4': 1.0, 'has_pattern': 1.5, 'has_material1': 1.0, 'has_material2': 1.0, 'created_by': 2.0, 'creator_birth_year': 1.0}

=== Seed Results ===
[chunk_신안22260] dense=0.6706
청자 향로 (소장품번호 신안22260) 중국의 원 시대 유물이다. 유물분류는 제례, 향로이다. 문양은 기하문이다. 재질은 도자기, 청자이다.

[chunk_신안20230] dense=0.6537
청자 향로 (소장품번호 신안20230) 중국의 원 시대 유물이다. 유물분류는 제례, 향로이다. 문양은 팔괘문이다. 재질은 도자기, 청자이다.

[chunk_신안20354] dense=0.6517
청자 향로 (소장품번호 신안20354) 중국의 원 시대 유물이다. 유물분류는 제례, 향로이다. 문양은 팔괘문이다. 재질은 도자기, 청자이다.

[chunk_신안21090] dense=0.6478
청자 향로 (소장품번호 신안21090) 중국의 원 시대 유물이다. 유물분류는 제례, 향로이다. 문양은 팔괘문이다. 재질은 도자기, 청자이다.

[chunk_신도1169] dense=0.6308
청자 팔괘무늬 향로 (소장품번호 신도1169) 중국의 원 시대 유물이다. 유물분류는 제례, 향로이다. 문양은 팔괘문이다. 재질은 도자기, 청자이다.


=== Final Hybrid Results ===
[chunk_동원2174] hybrid=0.6258 | dense=0.4935 | graph=1.0000 | rel_bonus=0.1000
「시경」의 「빈풍칠월편」 (소장품번호 동원2174) 한국의 조선 시대 유물이다. 유물분류는

In [31]:
query = "청자 향로의 재질은 무엇인가?"
out = hybrid_retrieve(query, seed_top_k=5, final_top_k=5, max_hop=2)

result = build_structured_answer(query, out, item_rel_values)

print(f"Question: {query}")
print(f"Top item: {result['top_item_id']}")
print(f"Target relation: {result['target_relation']}")
print(f"Answer: {result['answer']}")

Question: 청자 향로의 재질은 무엇인가?
Top item: 신안20230
Target relation: has_material1
Answer: 청자 향로의 재질은 도자기, 청자이다.


In [32]:
query = "이방운 작품에는 어떤 문양이 등장하는가?"
out = hybrid_retrieve(query, seed_top_k=5, final_top_k=5, max_hop=2)

result = build_structured_answer(query, out, item_rel_values)

print(f"Question: {query}")
print(f"Top item: {result['top_item_id']}")
print(f"Target relation: {result['target_relation']}")
print(f"Answer: {result['answer']}")

Question: 이방운 작품에는 어떤 문양이 등장하는가?
Top item: 동원2174
Target relation: has_pattern
Answer: 「시경」의 「빈풍칠월편」에 등장하는 문양은 나무, 산, 소, 화제, 말, 매미, 개, 버드나무, 새, 괴석, 사람, 돌이다.


In [33]:
query = "강남춘의도의 작가는 누구인가?"
out = hybrid_retrieve(query, seed_top_k=5, final_top_k=5, max_hop=2)

result = build_structured_answer(query, out, item_rel_values)

print(f"Question: {query}")
print(f"Top item: {result['top_item_id']}")
print(f"Target relation: {result['target_relation']}")
print(f"Answer: {result['answer']}")

Question: 강남춘의도의 작가는 누구인가?
Top item: 덕수1191-12
Target relation: created_by
Answer: 강남춘의도의 작가는 이인상이다.
